In [ ]:
import pandas as pd
import numpy as np
import logging
import sys

In [ ]:
LOG_FILE = 'load_clean_merge_test.log'
logger = logging.getLogger('load_clean_merge_test')
logger.setLevel(logging.DEBUG)
logger.propagate = False

if not logger.handlers:                          # guard against duplicate handlers on re-run
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(LOG_FILE)
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)

pd.set_option('display.max_columns', None)
logger.info(f"Libraries imported and logger configured. Full detail is being written to {LOG_FILE}")

2026-09-04 12:45:13,931 - load_clean_merge_test - INFO - Libraries imported and logger configured. Full detail is being written to load_clean_merge_test.log


**Loading the data**

In [ ]:
test = pd.read_csv('test.csv', parse_dates=['Date'])
logger.info(f"test.csv loaded. Shape: {test.shape}")

cleaned_store = pd.read_csv('cleaned_store.csv')
logger.info(f"cleaned_store.csv loaded. Shape: {cleaned_store.shape}")
logger.debug(f"test.csv columns: {test.columns.tolist()}")   # ← new line

test.head()

2026-09-04 12:46:59,887 - load_clean_merge_test - INFO - test.csv loaded. Shape: (41088, 8)
2026-09-04 12:46:59,894 - load_clean_merge_test - INFO - cleaned_store.csv loaded. Shape: (1115, 11)


,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
0,1,1,4,2015-09-17,1.0,1,0,0
1,2,3,4,2015-09-17,1.0,1,0,0
2,3,7,4,2015-09-17,1.0,1,0,0
3,4,8,4,2015-09-17,1.0,1,0,0
4,5,9,4,2015-09-17,1.0,1,0,0


* The output confirms that test.csv was loaded successfully with 41,088 rows and 8 columns.
The cleaned_store.csv file contains 1,115 rows and 11 columns.

* The displayed table shows the first five test records, including Store, DayOfWeek, Date, Open, Promo, and holiday information.

In [ ]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41088 entries, 0 to 41087
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Id             41088 non-null  int64         
 1   Store          41088 non-null  int64         
 2   DayOfWeek      41088 non-null  int64         
 3   Date           41088 non-null  datetime64[ns]
 4   Open           41077 non-null  float64       
 5   Promo          41088 non-null  int64         
 6   StateHoliday   41088 non-null  object        
 7   SchoolHoliday  41088 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(5), object(1)
memory usage: 2.5+ MB


* The output shows that the test dataset contains 41,088 records and 8 columns.
Most columns have complete data, but Open has 11 missing values.

* The dataset contains 5 integer columns, 1 float column, 1 date column, and 1 categorical column.

In [ ]:
# Missing Values
missing = test.isnull().sum()
logger.info("Missing values in test.csv:")
missing[missing > 0]

2026-09-04 12:47:29,552 - load_clean_merge_test - INFO - Missing values in test.csv:


,0
Open,11


* The output shows that the test dataset has 11 missing values in the Open column.
All other columns have no missing values.

In [ ]:
#Clean missing `Open` values
missing_open_rows = test[test['Open'].isnull()]
logger.info(f"Rows with missing Open: {len(missing_open_rows)}")
logger.info(f"Unique stores affected: {missing_open_rows['Store'].unique()}")
missing_open_rows[['Store', 'DayOfWeek', 'Date']]

2026-09-04 12:47:32,583 - load_clean_merge_test - INFO - Rows with missing Open: 11
2026-09-04 12:47:32,585 - load_clean_merge_test - INFO - Unique stores affected: [622]


,Store,DayOfWeek,Date
479,622,4,2015-09-17
1335,622,3,2015-09-16
2191,622,2,2015-09-15
3047,622,1,2015-09-14
4759,622,6,2015-09-12
5615,622,5,2015-09-11
6471,622,4,2015-09-10
7327,622,3,2015-09-09
8183,622,2,2015-09-08
9039,622,1,2015-09-07


* The output shows that 11 rows have missing values in the Open column.
* All 11 missing values belong to Store 622. These missing records occur across several dates from 5 to 17 September 2015.

In [ ]:
# Fill missing Open with 1
test['Open'] = test['Open'].fillna(1).astype(int)
logger.info("Filled missing Open values with 1 (assume store trading normally).")
logger.info(f"Remaining missing Open values: {test['Open'].isnull().sum()}")

2026-09-04 12:47:36,319 - load_clean_merge_test - INFO - Filled missing Open values with 1 (assume store trading normally).
2026-09-04 12:47:36,324 - load_clean_merge_test - INFO - Remaining missing Open values: 0


* The output shows that the 11 missing Open values were replaced with 1, assuming the stores were operating normally.
* After filling the values, the dataset has 0 remaining missing values in the Open column.
This confirms that the missing-value cleaning step was completed successfully.

**Data Type cleaning and Inconsistent values**

In [ ]:
logger.info(f"StateHoliday unique values BEFORE cleaning: {test['StateHoliday'].unique()}")
test['StateHoliday'] = test['StateHoliday'].astype(str)
test['StateHoliday'] = test['StateHoliday'].replace('0.0', '0')
logger.info(f"StateHoliday unique values AFTER cleaning: {test['StateHoliday'].unique()}")

2026-09-04 12:47:43,256 - load_clean_merge_test - INFO - StateHoliday unique values BEFORE cleaning: ['0' 'a']
2026-09-04 12:47:43,276 - load_clean_merge_test - INFO - StateHoliday unique values AFTER cleaning: ['0' 'a']


* The output shows that the StateHoliday column had two values, 0 and a, both before and after cleaning.
* No changes were required because there were no missing or invalid values.

In [ ]:
for col in ['Promo', 'SchoolHoliday', 'DayOfWeek', 'Open']:
    test[col] = test[col].astype(int)

logger.info("Confirmed Promo/SchoolHoliday/DayOfWeek/Open as integer dtypes.")
test.dtypes

2026-09-04 12:47:48,941 - load_clean_merge_test - INFO - Confirmed Promo/SchoolHoliday/DayOfWeek/Open as integer dtypes.


,0
Id,int64
Store,int64
DayOfWeek,int64
Date,datetime64[ns]
Open,int64
Promo,int64
StateHoliday,object
SchoolHoliday,int64


* The output confirms that Promo, SchoolHoliday, DayOfWeek, and Open were successfully converted to integer (int64) types.
* Date remains in the correct datetime format, while StateHoliday remains categorical (object).

In [ ]:
#Duplicate values
logger.info(f"Duplicate rows in test: {test.duplicated().sum()}")
logger.info(f"Duplicate (Store, Date) pairs: {test.duplicated(subset=['Store','Date']).sum()}")

2026-09-04 12:47:52,116 - load_clean_merge_test - INFO - Duplicate rows in test: 0
2026-09-04 12:47:52,121 - load_clean_merge_test - INFO - Duplicate (Store, Date) pairs: 0


* Duplicate rows = 0 - No completely duplicated records are present in the test dataset.
Duplicate (Store, Date) pairs = 0 - Each store has a unique date entry.
* This ensures there are no repeated records for the same store on the same day.
* Therefore, the test data has good record-level consistency and no duplicate removal is required.

**Merge test with cleaned_store**

In [ ]:
test_stores = set(test['Store'].unique())
store_ids = set(cleaned_store['Store'].unique())
missing_stores = test_stores - store_ids
logger.info(f"Store IDs in test.csv not found in store.csv: {len(missing_stores)}")
if missing_stores:
    logger.warning(f"Missing store IDs: {missing_stores}")

2026-09-04 12:48:04,043 - load_clean_merge_test - INFO - Store IDs in test.csv not found in store.csv: 0


* Store IDs not found = 0 - Every store ID in test.csv is also present in store.csv.
* This confirms that there are no unmatched or invalid store IDs.
* Therefore, the test dataset can be safely merged with the cleaned store data without losing store information.

In [ ]:
store_test = test.merge(cleaned_store, on='Store', how='left')
logger.info(f"Merge complete. store_test shape: {store_test.shape} (test had {test.shape[0]} rows)")

assert store_test.shape[0] == test.shape[0], "Row count changed after merge — check for duplicate Store rows in store.csv"
logger.info("Row count confirmed unchanged after merge.")

store_test.head()

2026-09-04 12:48:22,384 - load_clean_merge_test - INFO - Merge complete. store_test shape: (41088, 18) (test had 41088 rows)
2026-09-04 12:48:22,386 - load_clean_merge_test - INFO - Row count confirmed unchanged after merge.


,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,CompetitionOpenSinceKnown
0,1,1,4,2015-09-17,1,1,0,0,c,a,1270.0,9,2008,0,0,0,NoPromo2,1
1,2,3,4,2015-09-17,1,1,0,0,a,a,14130.0,12,2006,1,14,2011,"Jan,Apr,Jul,Oct",1
2,3,7,4,2015-09-17,1,1,0,0,a,c,24000.0,4,2013,0,0,0,NoPromo2,1
3,4,8,4,2015-09-17,1,1,0,0,a,a,7520.0,10,2014,0,0,0,NoPromo2,1
4,5,9,4,2015-09-17,1,1,0,0,a,c,2030.0,8,2000,0,0,0,NoPromo2,1


* This shows merge completed successfully, test.csv was merged with the cleaned store data.
* The merged store_test dataset contains 41,088 rows and 18 columns.
* The row count remained 41,088, exactly the same as the original test dataset.
* This confirms that the merge did not create duplicate or lost records.
* The sample shows that test information was successfully combined with store details such as StoreType, Assortment, CompetitionDistance, Promo2, and competition information.

In [ ]:
missing_after_merge = store_test.isnull().sum()
missing_after_merge[missing_after_merge > 0]

,0


Save merged store-test dataset

In [ ]:
csv_path = 'store_test.csv'
store_test.to_csv(csv_path, index=False)
logger.info(f"Saved merged store-test data to {csv_path}")
logger.info(f"Final store_test shape: {store_test.shape}")

2026-09-04 12:48:33,052 - load_clean_merge_test - INFO - Saved merged store-test data to store_test.csv
2026-09-04 12:48:33,053 - load_clean_merge_test - INFO - Final store_test shape: (41088, 18)


* Merged test data saved successfully as store_test.csv.
* The final dataset contains 41,088 rows and 18 columns.
* The shape confirms that no rows were lost or added during the merge.

In [ ]:
from google.colab import files

files.download(csv_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>